In [7]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from pathlib import Path
from matplotlib.patches import FancyArrowPatch

# ============================================================
# Minimal Haseman GIFs
# Output folder: figures/
# ============================================================

outdir = Path("figures")
outdir.mkdir(parents=True, exist_ok=True)

c = 1.0

def gamma(x):
    return 1 / (x - 1j*c)

def alpha_param(x, shift=0.58):
    return x + shift

def curve(xmin=0.06, xmax=5.5, n=900):
    x = np.linspace(xmin, xmax, n)
    z = gamma(x)
    return x, z

def fig_to_rgb(fig):
    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    rgba = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, 4)
    return rgba[:, :, :3].copy()

def clean_ax(ax, xlim=(-0.12, 0.88), ylim=(-0.70, 0.78), grid=False):
    ax.set_aspect("equal")
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    if grid:
        ax.grid(alpha=0.18, linestyle=":", linewidth=0.7)
    else:
        ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)

def orientation_arrow(ax, x0=2.8, dx=0.16):
    z1 = gamma(x0)
    z2 = gamma(x0 + dx)
    ax.add_patch(FancyArrowPatch(
        (z1.real, z1.imag), (z2.real, z2.imag),
        arrowstyle="-|>", mutation_scale=12, lw=1.2, color="black"
    ))



/home/enriquedo/.local/lib/python3.10/site-packages/matplotlib/animation.py:908: UserWarning: Animation was deleted without rendering anything. This is most likely not intended. To prevent deletion, assign the Animation to a variable, e.g. `anim`, that exists until you output the Animation using `plt.show()` or `anim.save()`.
  warnings.warn(


figures/haseman_shift_minimal.gif
figures/haseman_local_zoom_minimal.gif
figures/haseman_boundary_condition_minimal.gif
figures/haseman_reduction_diagram_minimal.gif


In [6]:
# ============================================================
# 1. Shift GIF
# ============================================================

def make_shift_gif(path=outdir / "haseman_shift_minimal.gif", frames=42):
    _, z = curve()
    xs = np.linspace(0.78, 2.55, frames)
    imgs = []

    for xt in xs:
        xa = alpha_param(xt)
        zt, za = gamma(xt), gamma(xa)
        zz = gamma(np.linspace(xt, xa, 120))

        fig, ax = plt.subplots(figsize=(7.2, 4.5), dpi=120)
        clean_ax(ax)

        ax.plot(z.real, z.imag, lw=3.0, color="#1f77b4")
        ax.plot(zz.real, zz.imag, lw=3.8, color="#8a6bbe", alpha=0.70)

        ax.plot(zt.real, zt.imag, "o", ms=7, color="#ff7f0e")
        ax.plot(za.real, za.imag, "o", ms=7, color="#2ca02c")

        orientation_arrow(ax)

        ax.text(0.10, 0.66, r"$\alpha:\Gamma_k\to\Gamma_k,\qquad t\mapsto\alpha(t)$",
                fontsize=17)
        ax.text(0.36, 0.49, "orientation", fontsize=12)

        ax.text(zt.real + 0.025, zt.imag - 0.060, r"$t$", fontsize=18)
        ax.text(za.real + 0.025, za.imag + 0.035, r"$\alpha(t)$", fontsize=18)
        ax.text(0.31, -0.58, r"$\Gamma_k$", fontsize=20)
        ax.text(0.07, -0.66, "The shift moves points along the same oriented branch.",
                fontsize=12)

        imgs.append(fig_to_rgb(fig))
        plt.close(fig)

    imageio.mimsave(path, imgs, duration=0.075, loop=0)
    return path



Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.6/317.6 KB 2.5 MB/s eta 0:00:00 MB/s eta 0:00:01:01


In [ ]:
# ============================================================
# 2. Local zoom GIF
# ============================================================

def make_local_zoom_gif(path=outdir / "haseman_local_zoom_minimal.gif", frames=42):
    _, z = curve()
    xt = 1.55
    zt = gamma(xt)

    left_start = zt + (-0.25 + 0.20j)
    right_start = zt + (0.27 + 0.02j)

    imgs = []
    svals = np.linspace(1.0, 0.04, frames)

    for s in svals:
        zl = zt + (left_start - zt) * s
        zr = zt + (right_start - zt) * s

        fig, ax = plt.subplots(figsize=(6.8, 4.8), dpi=120)
        clean_ax(ax, xlim=(-0.02, 0.84), ylim=(-0.34, 0.54), grid=True)

        # Keep light axes only in zoom
        for sp in ax.spines.values():
            sp.set_visible(True)
            sp.set_linewidth(0.8)
        ax.set_xticks(np.linspace(0, 0.8, 5))
        ax.set_yticks(np.linspace(-0.3, 0.5, 5))
        ax.tick_params(labelsize=8)

        ax.plot(z.real, z.imag, lw=3.0, color="#1f77b4")

        # approach rays
        ax.plot([left_start.real, zt.real], [left_start.imag, zt.imag],
                "--", lw=1.6, color="#9467bd", alpha=0.75)
        ax.plot([right_start.real, zt.real], [right_start.imag, zt.imag],
                "--", lw=1.6, color="#d62728", alpha=0.85)

        ax.plot(zt.real, zt.imag, "o", ms=7, color="#ff7f0e")
        ax.plot(zl.real, zl.imag, "o", ms=6, color="#7f7f7f")
        ax.plot(zr.real, zr.imag, "o", ms=6, color="#bcbd22")

        ax.annotate("", xy=(zt.real, zt.imag+0.10), xytext=(zt.real, zt.imag+0.02),
                    arrowprops=dict(arrowstyle="-|>", lw=1.2, color="black"))

        ax.text(0.20, 0.505, r"Local zoom at $t\in\Gamma$", fontsize=14)
        ax.text(zt.real + 0.015, zt.imag - 0.045, r"$t$", fontsize=17)
        ax.text(0.24, 0.16, "left side", fontsize=12)
        ax.text(0.55, 0.17, "right side", fontsize=12)
        ax.text(zt.real+0.015, zt.imag+0.105, "orientation", fontsize=10)
        ax.text(0.07, -0.27, r"$z_+(s)\to\alpha(t),\qquad z_-(s)\to t$",
                fontsize=12)

        imgs.append(fig_to_rgb(fig))
        plt.close(fig)

    imageio.mimsave(path, imgs, duration=0.075, loop=0)
    return path



In [ ]:
# ============================================================
# 3. Boundary condition GIF
# ============================================================

def make_boundary_gif(path=outdir / "haseman_boundary_condition_minimal.gif", frames=42):
    _, z = curve()
    xs = np.linspace(0.82, 2.35, frames)
    imgs = []

    for xt in xs:
        xa = alpha_param(xt)
        zt, za = gamma(xt), gamma(xa)

        left_start = za + (-0.20 + 0.18j)
        right_start = zt + (0.23 + 0.02j)

        fig, ax = plt.subplots(figsize=(7.0, 4.8), dpi=120)
        clean_ax(ax, xlim=(-0.10, 0.88), ylim=(-0.72, 0.88), grid=False)

        ax.plot(z.real, z.imag, lw=3.0, color="#1f77b4")
        ax.plot([left_start.real, za.real], [left_start.imag, za.imag],
                "--", lw=1.5, color="#9467bd")
        ax.plot([right_start.real, zt.real], [right_start.imag, zt.imag],
                "--", lw=1.5, color="#d62728")

        ax.plot(zt.real, zt.imag, "o", ms=7, color="#ff7f0e")
        ax.plot(za.real, za.imag, "o", ms=7, color="#2ca02c")

        orientation_arrow(ax)

        ax.text(-0.04, 0.76, r"$\Phi^+(\alpha(t))=G(t)\Phi^-(t)+g(t)$",
                fontsize=22)
        ax.text(0.22, 0.16, "left side", fontsize=12)
        ax.text(0.58, -0.08, "right side", fontsize=12)
        ax.text(zt.real + 0.025, zt.imag - 0.065, r"$t$", fontsize=18)
        ax.text(za.real + 0.025, za.imag + 0.035, r"$\alpha(t)$", fontsize=18)
        ax.text(0.20, -0.61, r"$\Gamma_k$", fontsize=19)
        ax.text(0.04, -0.68, r"Approach from the right side: $z_-(s)\to t$",
                fontsize=12)

        imgs.append(fig_to_rgb(fig))
        plt.close(fig)

    imageio.mimsave(path, imgs, duration=0.075, loop=0)
    return path



In [20]:
# ============================================================
# 4. Reduction diagram GIF
# ============================================================

def draw_reduction_frame(k, total_frames=120):
    fig, ax = plt.subplots(figsize=(8.2, 4.6), dpi=120)
    ax.axis("off")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    def box(x, y, txt, fs=15, show=True):
        if show:
            ax.text(
                x, y, txt,
                ha="center", va="center",
                fontsize=fs,
                bbox=dict(
                    boxstyle="round,pad=0.18",
                    fc="white",
                    ec="black",
                    lw=0.9
                )
            )

    def txt(x, y, s, fs=15, show=True):
        if show:
            ax.text(
                x, y, s,
                ha="center", va="center",
                fontsize=fs
            )

    def arrow(x1, y1, x2, y2, show=True):
        if show:
            ax.annotate(
                "",
                xy=(x2, y2),
                xytext=(x1, y1),
                arrowprops=dict(
                    arrowstyle="-|>",
                    lw=1.3,
                    color="black"
                )
            )

    # 6 etapas: 0, 1, 2, 3, 4, 5
    # Al aumentar total_frames, cada etapa dura más.
    stage = min(5, int(k / total_frames * 6))

    ax.text(
        0.5, 0.92,
        "Reduction to a singular integral equation",
        ha="center", va="center",
        fontsize=14,
        bbox=dict(
            boxstyle="round,pad=0.14",
            fc="white",
            ec="black",
            lw=0.8
        )
    )

    txt(
        0.50, 0.80,
        r"$\Phi^+(\alpha(t))=G(t)\Phi^-(t)+g(t)$",
        fs=20
    )

    box(0.13, 0.55, r"$\varphi$", fs=18, show=stage >= 0)

    arrow(0.18, 0.58, 0.29, 0.67, show=stage >= 1)
    arrow(0.18, 0.52, 0.29, 0.40, show=stage >= 1)
    box(0.24, 0.67, r"$P_+$", fs=12, show=stage >= 1)
    box(0.24, 0.40, r"$P_-$", fs=12, show=stage >= 1)

    box(0.35, 0.67, r"$P_+\varphi$", fs=17, show=stage >= 2)
    box(0.35, 0.40, r"$P_-\varphi$", fs=17, show=stage >= 2)

    arrow(0.43, 0.67, 0.55, 0.67, show=stage >= 3)
    arrow(0.43, 0.40, 0.55, 0.40, show=stage >= 3)
    box(0.50, 0.73, r"$V_\alpha$", fs=12, show=stage >= 3)
    box(0.50, 0.34, r"$G$", fs=12, show=stage >= 3)

    box(0.64, 0.67, r"$V_\alpha P_+\varphi$", fs=17, show=stage >= 4)
    box(0.64, 0.40, r"$GP_-\varphi$", fs=17, show=stage >= 4)

    arrow(0.72, 0.64, 0.79, 0.25, show=stage >= 5)
    arrow(0.72, 0.41, 0.79, 0.25, show=stage >= 5)
    box(
        0.79, 0.16,
        r"$(V_\alpha P_+ + GP_-)\varphi=g$",
        fs=18,
        show=stage >= 5
    )

    return fig


def make_reduction_gif(
    path=outdir / "haseman_reduction_diagram_minimal_slow.gif",
    frames=480,
    duration=50.0,
    final_pause_frames=25
):
    imgs = []

    for k in range(frames):
        fig = draw_reduction_frame(k, frames)
        imgs.append(fig_to_rgb(fig))
        plt.close(fig)

    # Pausa al final para que se pueda leer el resultado final
    if final_pause_frames > 0:
        last = imgs[-1]
        imgs.extend([last] * final_pause_frames)

    imageio.mimsave(
        path,
        imgs,
        duration=duration,
        loop=0
    )

    return path

In [21]:
paths = [
    make_shift_gif(),
    make_local_zoom_gif(),
    make_boundary_gif(),
    make_reduction_gif(),
]

for p in paths:
    print(p)

figures/haseman_shift_minimal.gif
figures/haseman_local_zoom_minimal.gif
figures/haseman_boundary_condition_minimal.gif
figures/haseman_reduction_diagram_minimal_slow.gif


In [22]:
# ============================================================
# Slow oscillation GIFs
# Output:
#   figures/slow_oscillation_near_zero.gif
#   figures/slow_oscillation_infinity.gif
# ============================================================

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from pathlib import Path
from matplotlib.patches import FancyBboxPatch

# ============================================================
# Output folder
# ============================================================

outdir = Path("figures")
outdir.mkdir(parents=True, exist_ok=True)

# ============================================================
# Utilities
# ============================================================

def fig_to_rgb(fig):
    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    rgba = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, 4)
    return rgba[:, :, :3].copy()


def save_gif(path, frames, duration=0.12, loop=0):
    imageio.mimsave(path, frames, duration=duration, loop=loop)


# ============================================================
# Model curve and slowly oscillating function
# ============================================================

def spiral(r, beta=1.15):
    """
    Logarithmic spiral:
        z(r) = r exp(i beta log r)
    """
    theta = beta * np.log(r)
    return r * np.exp(1j * theta)


def slow_function(r):
    """
    Example of a slowly oscillating function near 0 and infinity.
    The oscillations become slower in logarithmic scale.

    This is only for visualization.
    """
    return 0.55 + 0.25 * np.sin(np.log(1 + np.abs(np.log(r))))


def osc_on_window(r, mode="zero"):
    """
    Oscillation over a dyadic radial window.

    near zero:
        window [r, 2r]

    near infinity:
        window [r, 2r]
    """
    rr = np.linspace(r, 2 * r, 500)
    vals = slow_function(rr)
    return vals.max() - vals.min()


# ============================================================
# GIF 1: Slow oscillation near zero
# ============================================================

def make_slow_oscillation_near_zero_gif(
    path=outdir / "slow_oscillation_near_zero.gif",
    frames=70,
    duration=0.12,
):
    imgs = []

    # Spiral domain
    r_all = np.geomspace(0.008, 1.0, 2200)
    z_all = spiral(r_all)

    # Window moves to zero
    r_values = np.geomspace(0.35, 0.015, frames)

    # Precompute oscillation values
    osc_values = np.array([osc_on_window(r) for r in r_values])

    for i, r in enumerate(r_values):
        fig = plt.figure(figsize=(9.2, 4.8), dpi=130)

        # Two panels
        ax1 = fig.add_axes([0.06, 0.13, 0.44, 0.74])
        ax2 = fig.add_axes([0.57, 0.16, 0.36, 0.68])

        # -------------------------
        # Left: spiral and dyadic window
        # -------------------------
        ax1.set_aspect("equal")
        ax1.axis("off")

        ax1.plot(
            z_all.real,
            z_all.imag,
            lw=2.2,
            color="#1f77b4",
            alpha=0.18
        )

        # Window [r, 2r]
        mask = (r_all >= r) & (r_all <= 2 * r)
        z_win = z_all[mask]

        ax1.plot(
            z_win.real,
            z_win.imag,
            lw=4.0,
            color="#1f77b4"
        )

        # endpoints of the window
        z_r = spiral(r)
        z_2r = spiral(2 * r)

        ax1.plot(z_r.real, z_r.imag, "o", ms=6, color="#ff7f0e")
        ax1.plot(z_2r.real, z_2r.imag, "o", ms=6, color="#2ca02c")

        ax1.text(
            -0.92, 0.86,
            r"Slow oscillation near $0$",
            fontsize=15,
            color="darkgreen"
        )

        ax1.text(
            -0.92, 0.72,
            r"window $[r,2r]$",
            fontsize=13
        )

        ax1.text(
            -0.92, -0.84,
            r"$r\to 0$",
            fontsize=16
        )

        ax1.set_xlim(-1.05, 1.05)
        ax1.set_ylim(-1.00, 1.00)

        # -------------------------
        # Right: oscillation graph
        # -------------------------
        ax2.grid(alpha=0.20, linestyle=":", linewidth=0.8)

        ax2.plot(
            r_values[:i+1],
            osc_values[:i+1],
            lw=2.4,
            color="#1f77b4"
        )

        ax2.plot(
            r_values[i],
            osc_values[i],
            "o",
            ms=6,
            color="#ff7f0e"
        )

        ax2.set_xscale("log")
        ax2.invert_xaxis()

        ax2.set_title(r"$\Delta(r)=\operatorname{osc}(f,[r,2r])$",
                      fontsize=12)

        ax2.set_xlabel(r"$r\to 0$", fontsize=11)
        ax2.set_ylabel(r"$\Delta(r)$", fontsize=11)

        ymax = max(0.04, osc_values.max() * 1.25)
        ax2.set_ylim(0, ymax)

        ax2.text(
            0.05, 0.86,
            r"$\Delta(r)\to 0$",
            transform=ax2.transAxes,
            fontsize=13,
            bbox=dict(
                boxstyle="round,pad=0.16",
                fc="white",
                ec="darkgreen",
                lw=0.8
            )
        )

        imgs.append(fig_to_rgb(fig))
        plt.close(fig)

    # final pause
    imgs.extend([imgs[-1]] * 18)

    save_gif(path, imgs, duration=duration)
    return path


# ============================================================
# GIF 2: Slow oscillation at infinity
# ============================================================

def make_slow_oscillation_infinity_gif(
    path=outdir / "slow_oscillation_infinity.gif",
    frames=70,
    duration=0.12,
):
    imgs = []

    # Spiral domain
    r_all = np.geomspace(0.15, 25.0, 2500)
    z_all = spiral(r_all)

    # Normalize for display
    z_plot = z_all / np.max(np.abs(z_all))

    # Window moves to infinity
    r_values = np.geomspace(0.4, 10.0, frames)
    osc_values = np.array([osc_on_window(r) for r in r_values])

    for i, r in enumerate(r_values):
        fig = plt.figure(figsize=(9.2, 4.8), dpi=130)

        ax1 = fig.add_axes([0.06, 0.13, 0.44, 0.74])
        ax2 = fig.add_axes([0.57, 0.16, 0.36, 0.68])

        # -------------------------
        # Left: spiral and dyadic window
        # -------------------------
        ax1.set_aspect("equal")
        ax1.axis("off")

        ax1.plot(
            z_plot.real,
            z_plot.imag,
            lw=2.2,
            color="#1f77b4",
            alpha=0.18
        )

        mask = (r_all >= r) & (r_all <= 2 * r)
        z_win = z_all[mask] / np.max(np.abs(z_all))

        ax1.plot(
            z_win.real,
            z_win.imag,
            lw=4.0,
            color="#1f77b4"
        )

        z_r = spiral(r) / np.max(np.abs(z_all))
        z_2r = spiral(2 * r) / np.max(np.abs(z_all))

        ax1.plot(z_r.real, z_r.imag, "o", ms=6, color="#ff7f0e")
        ax1.plot(z_2r.real, z_2r.imag, "o", ms=6, color="#2ca02c")

        ax1.text(
            -0.92, 0.86,
            r"Slow oscillation at $\infty$",
            fontsize=15,
            color="darkgreen"
        )

        ax1.text(
            -0.92, 0.72,
            r"window $[r,2r]$",
            fontsize=13
        )

        ax1.text(
            -0.92, -0.84,
            r"$r\to\infty$",
            fontsize=16
        )

        ax1.set_xlim(-1.05, 1.05)
        ax1.set_ylim(-1.00, 1.00)

        # -------------------------
        # Right: oscillation graph
        # -------------------------
        ax2.grid(alpha=0.20, linestyle=":", linewidth=0.8)

        ax2.plot(
            r_values[:i+1],
            osc_values[:i+1],
            lw=2.4,
            color="#1f77b4"
        )

        ax2.plot(
            r_values[i],
            osc_values[i],
            "o",
            ms=6,
            color="#ff7f0e"
        )

        ax2.set_xscale("log")

        ax2.set_title(r"$\Delta(r)=\operatorname{osc}(f,[r,2r])$",
                      fontsize=12)

        ax2.set_xlabel(r"$r\to\infty$", fontsize=11)
        ax2.set_ylabel(r"$\Delta(r)$", fontsize=11)

        ymax = max(0.04, osc_values.max() * 1.25)
        ax2.set_ylim(0, ymax)

        ax2.text(
            0.05, 0.86,
            r"$\Delta(r)\to 0$",
            transform=ax2.transAxes,
            fontsize=13,
            bbox=dict(
                boxstyle="round,pad=0.16",
                fc="white",
                ec="darkgreen",
                lw=0.8
            )
        )

        imgs.append(fig_to_rgb(fig))
        plt.close(fig)

    imgs.extend([imgs[-1]] * 18)

    save_gif(path, imgs, duration=duration)
    return path


# ============================================================
# Run
# ============================================================

near_zero_path = make_slow_oscillation_near_zero_gif()
infinity_path = make_slow_oscillation_infinity_gif()

print("Generated:")
print(near_zero_path)
print(infinity_path)

Generated:
figures/slow_oscillation_near_zero.gif
figures/slow_oscillation_infinity.gif
